In [4]:
import pandas as pd
import numpy as np

# Load the raw dataset again for cross-validation
df = pd.read_csv('ardd_fatalities.csv', low_memory=False)

if 'Unnamed: 23' in df.columns:
    df = df.drop(columns=['Unnamed: 23'])

# Isolate target age group
df['Age'] = df['Age'].replace([-9, '-9', '-9.0', 'Unknown'], np.nan)
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
df = df.dropna(subset=['Age'])
df = df[df['Age'] >= 17].copy()

# Set up target variable
df['Target_Youth'] = (df['Age'] <= 25).astype(int)

# Select the original features 
feature_cols = [
    'State', 'Speed Limit', 'Crash Type', 'Bus Involvement', 
    'Heavy Rigid Truck Involvement', 'Articulated Truck Involvement', 
    'Dayweek', 'Time of day'
]

# Create X and y variables so the cross-validation can read them!
X = df[feature_cols].copy()
y = df['Target_Youth'].values


In [2]:
%pip install scikit-elcs

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# =========================================================
# TASK 5: STRATIFIED K-FOLD CV & STATISTICAL SIGNIFICANCE
# =========================================================

import numpy as np
import time
from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from imblearn.over_sampling import SMOTE
from skeLCS import eLCS
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, balanced_accuracy_score
from scipy import stats

# Assuming X (raw features with string values/sentinels) and y (Target_Youth) 
# are carried over from clean Task 3 initialization steps.

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Lists to store metrics across folds
baseline_f1_scores = []
improved_f1_scores = []
baseline_acc_scores = []
improved_acc_scores = []

fold = 1
for train_idx, test_idx in skf.split(X, y):
    print(f"--- Processing Cross-Validation Fold {fold}/5 ---")
    
    # Split raw data
    X_train_fold, X_test_fold = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
    y_train_fold, y_test_fold = y[train_idx], y[test_idx]
    
    # ---------------------------------------------------
    # PIPELINE A: Baseline Data Processing (Simple encoding only)
    # ---------------------------------------------------
    X_b_train = X_train_fold.copy()
    X_b_test = X_test_fold.copy()
    for col in X_b_train.columns:
        X_b_train[col] = X_b_train[col].astype(str).astype('category').cat.codes
        X_b_test[col] = X_b_test[col].astype(str).astype('category').cat.codes
        
    # ---------------------------------------------------
    # PIPELINE B: Improved Data Processing (No-Leakage Imputation + SMOTE)
    # ---------------------------------------------------
    cat_cols = X_train_fold.select_dtypes(include=['object', 'category']).columns.tolist()
    num_cols = X_train_fold.select_dtypes(include=['int64', 'float64']).columns.tolist()
    
    # Impute and encode categories
    imputer_cat = SimpleImputer(strategy='most_frequent')
    X_train_fold[cat_cols] = imputer_cat.fit_transform(X_train_fold[cat_cols])
    X_test_fold[cat_cols] = imputer_cat.transform(X_test_fold[cat_cols])
    
    encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X_train_fold[cat_cols] = encoder.fit_transform(X_train_fold[cat_cols])
    X_test_fold[cat_cols] = encoder.transform(X_test_fold[cat_cols])
    
    # Impute numeric columns
    if num_cols:
        imputer_num = SimpleImputer(strategy='median')
        X_train_fold[num_cols] = imputer_num.fit_transform(X_train_fold[num_cols])
        X_test_fold[num_cols] = imputer_num.transform(X_test_fold[num_cols])
        
    X_train_clean = X_train_fold.values.astype(np.float32)
    X_test_clean = X_test_fold.values.astype(np.float32)
    
    # Balance training fold via SMOTE
    smote = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train_clean, y_train_fold)
    
    # ---------------------------------------------------
    # MODEL TRAINING & EVALUATION
    # ---------------------------------------------------
    # Train Baseline Model
    model_base = eLCS(learning_iterations=5000, N=500, p_spec=0.5)
    model_base.fit(X_b_train.values, y_train_fold)
    pred_base = model_base.predict(X_b_test.values)
    
    # Train Improved Model
    model_imp = eLCS(learning_iterations=5000, N=500, p_spec=0.5)
    model_imp.fit(X_train_resampled, y_train_resampled)
    pred_imp = model_imp.predict(X_test_clean)
    
    # Save target score outcomes
    baseline_f1_scores.append(f1_score(y_test_fold, pred_base, average='weighted'))
    improved_f1_scores.append(f1_score(y_test_fold, pred_imp, average='weighted'))
    baseline_acc_scores.append(accuracy_score(y_test_fold, pred_base))
    improved_acc_scores.append(accuracy_score(y_test_fold, pred_imp))
    
    fold += 1

# ---------------------------------------------------
# STATISTICAL HYPOTHESIS TESTING (Paired t-test)
# ---------------------------------------------------
t_stat_f1, p_val_f1 = stats.ttest_rel(baseline_f1_scores, improved_f1_scores)

print("\n" + "="*55)
print("             TASK 5 STATISTICAL EVALUATION")
print("="*55)
print(f"Baseline Mean F1-Score: {np.mean(baseline_f1_scores):.4f}")
print(f"Improved Mean F1-Score: {np.mean(improved_f1_scores):.4f}")
print(f"Paired t-test Statistic (F1): {t_stat_f1:.4f}")
print(f"Calculated p-value (F1):      {p_val_f1:.6f}")
print("-" * 55)
if p_val_f1 < 0.05:
    print("RESULT: Statistically Significant (Reject H0, p < 0.05)")
else:
    print("RESULT: Not Statistically Significant (Fail to Reject H0)")
print("="*55)


--- Processing Cross-Validation Fold 1/5 ---


C:\Users\ASUS F15\AppData\Local\Temp\ipykernel_63848\1081678580.py:46: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train_fold.select_dtypes(include=['object', 'category']).columns.tolist()


--- Processing Cross-Validation Fold 2/5 ---


C:\Users\ASUS F15\AppData\Local\Temp\ipykernel_63848\1081678580.py:46: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train_fold.select_dtypes(include=['object', 'category']).columns.tolist()


--- Processing Cross-Validation Fold 3/5 ---


...

--- Processing Cross-Validation Fold 4/5 ---


C:\Users\ASUS F15\AppData\Local\Temp\ipykernel_63848\1081678580.py:46: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train_fold.select_dtypes(include=['object', 'category']).columns.tolist()


--- Processing Cross-Validation Fold 5/5 ---


C:\Users\ASUS F15\AppData\Local\Temp\ipykernel_63848\1081678580.py:46: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train_fold.select_dtypes(include=['object', 'category']).columns.tolist()



             TASK 5 STATISTICAL EVALUATION
Baseline Mean F1-Score: 0.6047
Improved Mean F1-Score: 0.5959
Paired t-test Statistic (F1): 0.5944
Calculated p-value (F1):      0.584232
-------------------------------------------------------
RESULT: Not Statistically Significant (Fail to Reject H0)
